In [1]:
library(Seurat)
library(Signac)
library(GenomeInfoDb)
library(EnsDb.Hsapiens.v86)
library(ggplot2)
library(patchwork)
library(hdf5r)
library(future)
library(RColorBrewer)
library(dplyr)
library(Matrix)
library(BSgenome.Hsapiens.UCSC.hg38)
library(glue)
library(harmony)
library(matrixStats)
library(scales)
library(biomaRt)
library(curl)
library(goseq)
library(httr)
library(Scillus)
library(TFBSTools)
library(JASPAR2020)
library(ggridges)
library(ggrepel)
library(ggsignif)
library(qusage)
library(tidyverse)
httr::set_config(config(ssl_verifypeer = 0L))
set.seed(1234)
setwd("/home/jupyter/scATAC_analysis/edit/snatac-rcc-manuscript")
source("scripts/functions.r")


Attaching SeuratObject

Loading required package: BiocGenerics


Attaching package: ‘BiocGenerics’


The following objects are masked from ‘package:stats’:

    IQR, mad, sd, var, xtabs


The following objects are masked from ‘package:base’:

    anyDuplicated, aperm, append, as.data.frame, basename, cbind,
    colnames, dirname, do.call, duplicated, eval, evalq, Filter, Find,
    get, grep, grepl, intersect, is.unsorted, lapply, Map, mapply,
    match, mget, order, paste, pmax, pmax.int, pmin, pmin.int,
    Position, rank, rbind, Reduce, rownames, sapply, setdiff, sort,
    table, tapply, union, unique, unsplit, which.max, which.min


Loading required package: S4Vectors

Loading required package: stats4


Attaching package: ‘S4Vectors’


The following object is masked from ‘package:utils’:

    findMatches


The following objects are masked from ‘package:base’:

    expand.grid, I, unname


Loading required package: IRanges

Loading required package: ensembldb

Loading required packag

# Table S1

## Sheet A: Clinical data for the internal scATAC-seq cohort

Read in object metadata

In [ ]:
all_metadata <- readRDS("processed_data/allcells_seurat_metadata.rds")
metadata <- all_metadata %>% filter(broad_celltype_excluded != "Excluded")


Add in sample preservation type

In [ ]:
metadata <- metadata %>%
  mutate(
    sample_status = case_when(
      grepl("multiome", sample) ~ "Frozen",
      cohort %in% c("in-house", "wuetal") ~ "Frozen",
      cohort %in% c("longetal", "yuetal") ~ "Fresh",
      TRUE ~ "Unknown"
    )
  )


Add dbgap IDs

In [ ]:
# add in dbgap subject and sample IDs for internal cohort
metadata$cell_barcodes <- row.names(metadata)
## subject ID
dbgap_subjectID_mapping <- read.table("processed_data/internal_participant_to_dbgapsubjectid_mapping.txt", sep = "\t", header = TRUE)
cell_metadata <- dplyr::left_join(metadata, dbgap_subjectID_mapping, by = "participant")
## sample ID
original_sampleids <- unique((cell_metadata %>% filter(cohort == "in-house"))$sample)
predicted_dbgap_sampleids_v <- ifelse(grepl("multiome", original_sampleids), paste0(original_sampleids, "_ATAC"), original_sampleids)
predicted_dbgap_sampleids <- data.frame(sample = original_sampleids, dbgap_sampleid = predicted_dbgap_sampleids_v)
cell_metadata <- dplyr::left_join(cell_metadata, predicted_dbgap_sampleids, by = "sample")


Format and save

In [ ]:
# sample/biopsy level metadata for internal cohort (newly generated data)
comut_cols <- c(
  "sample", "biopsy", "participant", "dbgap_subjectid", "dbgap_sampleid", "male", "age_at_diagnosis",
  "fuhrman_grade", "stage", "histology", "sarcomatoid_features", "biopsy_site", "sample_status"
)
internal_cohort_sample_meta <- cell_metadata %>%
  filter(cohort == "in-house") %>%
  dplyr::select(comut_cols) %>%
  distinct()
internal_cohort_sample_meta <- internal_cohort_sample_meta %>%
  rename(
    Sex = male,
    Age = age_at_diagnosis,
    Grade = fuhrman_grade,
    Stage = stage,
    Histology = histology,
    `Sarcomatoid features` = sarcomatoid_features,
    `Biopsy site` = biopsy_site,
    Sample = sample,
    Biopsy = biopsy,
    Participant = participant,
    `dbGaP Subject ID` = dbgap_subjectid,
    `dbGaP Sample ID` = dbgap_sampleid,
    `Sample preservation method` = sample_status
  ) %>%
  mutate(
    Sex = ifelse(Sex == 1, "Male",
      ifelse(Sex == 0, "Female", NA)
    ),
    `Sarcomatoid features` = case_when(
      `Sarcomatoid features` == 1 ~ "Yes",
      `Sarcomatoid features` == 0 ~ "No",
      is.na(`Sarcomatoid features`) ~ "Not evaluated"
    )
  )

write.table(internal_cohort_sample_meta, file = "tables/s1a_internal_cohort_sample_metadata.txt", sep = "\t", quote = FALSE, row.names = FALSE, col.names = TRUE)


## Sheet B: Non-silent PBRM1, BAP1, KDM5C, SETD2 mutations in the internal cohort

In [ ]:
# Read in all samples mutation data
muts <- read.table("processed_data/allcohorts_mutation_data.txt", sep = "\t")
muts <- muts %>% filter(value != "Unable to determine")
muts <- muts %>% mutate(pLOF = case_when(
  value %in% c("Missense", "In frame indel") ~ 0,
  TRUE ~ 1
))
colnames(muts) <- c("gene", "consequence", "biopsy", "pLOF")

# add in sample and participant ID for easy mapping
mapping <- readRDS("processed_data/allcells_seurat_metadata.rds")
muts_w_ids <- dplyr::inner_join(muts, mapping %>% dplyr::select(c("biopsy", "sample", "participant", "cohort")) %>% distinct(), by = "biopsy")
muts_w_ids <- muts_w_ids %>%
  filter(cohort == "in-house") %>%
  dplyr::select(c("participant", "biopsy", "sample", "gene", "consequence", "pLOF"))
muts_w_ids <- muts_w_ids %>%
  rename(
    Participant = participant,
    Biopsy = biopsy,
    Sample = sample,
    Gene = gene,
    Consequence = consequence
  )
# three samples have no mutation info c('rcc_0600855_T2', 'rcc_0600920_T1','rcc_RCCT1324-T1A'). the other samples can safely assume if mutation not here then its not present
write.table(muts_w_ids, "tables/s1b_mutations.txt", sep = "\t", quote = F, row.names = F, col.names = T)


Warning message in dplyr::inner_join(muts, mapping %>% dplyr::select(c("biopsy", :
"Detected an unexpected many-to-many relationship between `x` and `y`.
ℹ Row 1 of `x` matches multiple rows in `y`.
ℹ Row 26 of `y` matches multiple rows in `x`.
ℹ If a many-to-many relationship is expected, set `relationship =
  "many-to-many"` to silence this warning."


## Sheet C: Cell level copy number data, QC metrics, cell type annotations across all scATAC-seq cohorts

In [ ]:
chrall <- grep("chr", colnames(cell_metadata), value = TRUE)
comut_cols <- c(
  "cell_barcodes", "sample", "biopsy", "participant", "doublet_score",
  "broad_celltype_excluded", chrall, "blacklist_fraction", "nucleosome_signal",
  "TSS.enrichment", "FRiP", "peak_region_fragments", "cohort"
)
length(comut_cols)
cell_metadata_supp <- cell_metadata %>% dplyr::select(comut_cols)
# re-map cell values where desired
cell_metadata_supp <- cell_metadata_supp %>% mutate(
  cohort = case_when(
    cohort == "wuetal" ~ "Wu et al Nat Comm 2023",
    cohort == "longetal" ~ "Long et al Cell Discov 2022",
    cohort == "yuetal" ~ "Yu et al Cancer Res 2023",
    cohort == "in-house" ~ "Internal"
  )
)

cell_metadata_supp <- cell_metadata_supp %>%
  rename(
    `Cell barcode` = cell_barcodes,
    Sample = sample,
    Biopsy = biopsy,
    Participant = participant,
    `Doublet score` = doublet_score,
    `Broad cell type` = broad_celltype_excluded,
    Sample = sample,
    Biopsy = biopsy,
    Participant = participant,
    `Blacklist fraction` = blacklist_fraction,
    `Nucleosome signal` = nucleosome_signal,
    `TSS enrichment` = TSS.enrichment,
    `Peak region fragments (nCount_ATAC, sequencing depth)` = peak_region_fragments
  )
dim(cell_metadata_supp)
head(cell_metadata_supp)
write.table(cell_metadata_supp, file = "tables/s1c_cell_metadata.txt", sep = "\t", quote = FALSE, row.names = FALSE, col.names = TRUE)


[1] 56

[1] 177845     56

,Cell barcode,Sample,Biopsy,Participant,Doublet score,Broad cell type,chr10p,chr10q,chr11p,chr11q,⋯,chrYq,chrXp,chr21p,chr22p,Blacklist fraction,Nucleosome signal,TSS enrichment,FRiP,"Peak region fragments (nCount_ATAC, sequencing depth)",cohort
,<chr>,<chr>,<chr>,<chr>,<dbl>,<fct>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
1,C3L-00004-01_GGGCCATGTTCTACCC-1,C3L-00004-01,C3L-00004-01,C3L-00004,0.123358443,Tumor,1.845502,1.84041,1.819084,1.697953,⋯,1.793084,NA,NA,NA,0.008461771,0.5299340,5.189575,0.6240100,29405.81,Wu et al Nat Comm 2023
2,C3L-00004-01_TCACCACGTACAAGCG-1,C3L-00004-01,C3L-00004-01,C3L-00004,0.135753304,Tumor,2.762440,1.84041,1.819084,1.697953,⋯,1.793084,NA,NA,NA,0.006627219,0.5445879,5.637728,0.7754665,21505.90,Wu et al Nat Comm 2023
3,C3L-00004-01_TCCCACACACGGTTAT-1,C3L-00004-01,C3L-00004-01,C3L-00004,0.003836718,Perivascular cell,2.762440,1.84041,1.819084,1.697953,⋯,1.793084,NA,NA,NA,0.006557855,0.5327389,4.537974,0.5889876,27165.67,Wu et al Nat Comm 2023
4,C3L-00004-01_CCCTCTCGTAAAGGCC-1,C3L-00004-01,C3L-00004-01,C3L-00004,0.118392594,Tumor,1.845502,1.84041,1.819084,1.697953,⋯,1.793084,NA,NA,NA,0.007877148,0.5368630,5.393919,0.6296278,23747.05,Wu et al Nat Comm 2023
5,C3L-00004-01_TAGGAGGCAATGTAAG-1,C3L-00004-01,C3L-00004-01,C3L-00004,0.009539410,Tumor,1.845502,1.84041,1.819084,1.697953,⋯,1.793084,NA,NA,NA,0.007144404,0.3746113,6.469055,0.8635797,21806.31,Wu et al Nat Comm 2023
6,C3L-00004-01_GCATTGACACCTGTGG-1,C3L-00004-01,C3L-00004-01,C3L-00004,0.002748405,Tumor,1.845502,1.84041,1.819084,1.697953,⋯,1.793084,NA,NA,NA,0.006280843,0.5012121,6.126800,0.8416614,14113.13,Wu et al Nat Comm 2023


## Sheet D: Significantly differentially accessible genes between each broad non-tumor cell type and all other non-tumor cell types

In [ ]:
# read in unfiltered differentially accessible gene output
dag <- readRDS("processed_data/nontumor_dags.rds")
# filter to significant and save
write.table(dag %>% filter(p_val_adj < 0.05), file = "tables/s1d_nontumor_diffgenes.txt", sep = "\t", quote = F, col.names = T, row.names = F)


# Export to Excel
Switch to python kernel. 

In [1]:
import pandas as pd
import os
os.chdir('/home/jupyter/scATAC_analysis/edit/snatac-rcc-manuscript')

In [3]:
sample_metadata = pd.read_csv("tables/s1a_internal_cohort_sample_metadata.txt", sep = "\t")
print(sample_metadata.head())

mutations = pd.read_csv("tables/s1b_mutations.txt", sep = "\t")
print(mutations.head())

cell_metadata = pd.read_csv("tables/s1c_cell_metadata.txt", sep = "\t")
print(cell_metadata.head())

dags = pd.read_csv("tables/s1d_nontumor_diffgenes.txt", sep = "\t")
print(dags.head())


                            Sample          Biopsy Participant  \
0  CCG1114e_0600855_T3_B1_multiome  rcc_0600855_T3     0600855   
1  CCG1114e_0600876_T1_G1_multiome  rcc_0600876_T1     0600876   
2  CCG1114e_0600913_T1_E1_multiome  rcc_0600913_T1     0600913   
3  CCG1114e_0600915_T1_H3_multiome  rcc_0600915_T1     0600915   
4                   rcc_0600855_T1  rcc_0600855_T1     0600855   

  dbGaP Subject ID                       dbGaP Sample ID     Sex  Age  Grade  \
0       0600855_T3  CCG1114e_0600855_T3_B1_multiome_ATAC    Male   57      3   
1       0600876_T1  CCG1114e_0600876_T1_G1_multiome_ATAC  Female   68      4   
2       0600913_T1  CCG1114e_0600913_T1_E1_multiome_ATAC    Male   47      4   
3       0600915_T1  CCG1114e_0600915_T1_H3_multiome_ATAC    Male   58      3   
4       0600855_T3                        rcc_0600855_T1    Male   57      3   

   Stage   Histology Sarcomatoid features Biopsy site  \
0      4  Clear cell        Not evaluated     Abdomen   
1      4

/opt/conda/lib/python3.7/site-packages/IPython/core/interactiveshell.py:3524: DtypeWarning: Columns (3) have mixed types.Specify dtype option on import or set low_memory=False.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [4]:
with pd.ExcelWriter('tables/table_S1_draft.xlsx') as writer:  
    sample_metadata.to_excel(writer, sheet_name='A', index = False)
    mutations.to_excel(writer, sheet_name='B', index = False)
    cell_metadata.to_excel(writer, sheet_name='C', index = False)
    dags.to_excel(writer, sheet_name='D', index = False)

# Then add in README with titles and save as non-draft